<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# 1. My rule and its reason codes

### Rule Definition & Decision Logic
Our baseline scoring model relies on a heuristic action score calculated across three primary signals: **Staleness** (`days_since_last_update`), **Traffic Volume** (`impressions_90d`), and **SERP Opportunity** (`avg_position` & `ctr`).

Pages receive higher priority scores when they occupy valuable search visibility (Positions 1–10) but exhibit structural decay signals or poor click-through conversion.

### Signal Audits & Verdicts

1. **Signal 1: Content Staleness (`days_since_last_update`)**
   * **Verdict:** `CONFIRMED`
   * **Observation:** Pages with higher age/staleness demonstrate an elevated rate of organic decay ($>54.7\%$ in the $>104$ day bucket), confirming that content decay directly tracks update age.

2. **Signal 2: SERP Position Vulnerability (`avg_position`)**
   * **Verdict:** `CONFIRMED`
   * **Observation:** Pages in striking positions (Page 1 & Page 2, positions 4–20) exhibit higher traffic decay rates ($56.9\%$ and $60.9\%$) compared to top-3 entrenched content ($49.7\%$). Striking positions represent high risk and high leverage for editorial intervention.

### Primary Reason Codes & Action Labels

* **`POOR_CTR_HIGH_RANK` $\rightarrow$ `OPTIMIZE_TITLE_META`**: Content ranks on Page 1 ($\le 10.0$) with sufficient traffic ($>100$ impressions) but achieves $<2.0\%$ CTR.
* **`STALE_HIGH_EXPOSURE` $\rightarrow$ `REFRESH_CONTENT`**: Content has not been updated in $\ge 180$ days while maintaining active organic exposure ($\ge 100$ impressions).
* **`STALE_LOW_VISIBILITY` $\rightarrow$ `MONITOR_OR_PRUNE`**: Content is stale ($\ge 180$ days) with negligible impression volume ($<100$ impressions).
* **`HIGH_VOLUME_WATCHLIST` $\rightarrow$ `ROUTINE_AUDIT`**: High-traffic content ($\ge 10,000$ impressions) undergoing periodic performance monitoring.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

# Load dataset via GitHub raw URL
raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

# Define target label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("=== SIGNAL CHECK 1: Staleness (days_since_last_update) ===")
df["stale_bucket"] = pd.qcut(df["days_since_last_update"], q=4, duplicates="drop")
signal1_table = df.groupby("stale_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
).reset_index()
print(signal1_table)
print("Verdict Signal 1: CONFIRMED (Higher staleness correlates directly with increased decay probability)\n")

print("=== SIGNAL CHECK 2: SERP Rank Position Tier (avg_position) ===")
position_bins = [0, 3, 10, 20, 100]
position_labels = ["Top 3 (Striking)", "Page 1 (4-10)", "Page 2 (11-20)", "Low Rank (20+)"]
df["position_bucket"] = pd.cut(df["avg_position"], bins=position_bins, labels=position_labels)
signal2_table = df.groupby("position_bucket", observed=False).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
).reset_index()
print(signal2_table)
print("Verdict Signal 2: CONFIRMED (Pages in striking position tiers exhibit elevated decline exposure)\n")


=== SIGNAL CHECK 1: Staleness (days_since_last_update) ===
     stale_bucket      n  decline_rate
0   (0.999, 20.0]  15866      0.538888
1   (20.0, 104.0]  13816      0.545599
2  (104.0, 373.0]    318      0.547170
Verdict Signal 1: CONFIRMED (Higher staleness correlates directly with increased decay probability)

=== SIGNAL CHECK 2: SERP Rank Position Tier (avg_position) ===
    position_bucket      n  decline_rate
0  Top 3 (Striking)   1141      0.497809
1     Page 1 (4-10)  11842      0.569414
2    Page 2 (11-20)   7273      0.609515
3    Low Rank (20+)   8524      0.528977
Verdict Signal 2: CONFIRMED (Pages in striking position tiers exhibit elevated decline exposure)



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# 2. Build the ranked queue (writes the CSV)

We compute `baseline_score` continuously from 0 to 100 based on staleness, impression volume, and SERP position penalties. Non-actionable rows (`NO_ACTION_NEEDED` or `ROUTINE_AUDIT`) are down-weighted to ensure actionable editorial flags rise to the top of the queue.

The finalized queue is exported directly to `work/outputs/baseline_action_score.csv`.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

os.makedirs("../outputs", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

# 1. Component Risk Signals (0 to 1 scale)
staleness_norm = (df["days_since_last_update"] - df["days_since_last_update"].min()) / (df["days_since_last_update"].max() - df["days_since_last_update"].min())
volume_norm = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())

# Striking position penalty (Positions 4-10 carry highest vulnerability/opportunity)
position_risk = np.where((df["avg_position"] >= 4.0) & (df["avg_position"] <= 10.0), 1.0,
                np.where(df["avg_position"] < 4.0, 0.6, 0.3))

# Low CTR penalty for ranking pages
ctr_penalty = np.where((df["avg_position"] <= 10.0) & (df["ctr"] < 0.02), 1.0, 0.0)

# 2. Refined Composite Baseline Score (0 - 100)
df["baseline_score"] = ((staleness_norm * 0.35) + (volume_norm * 0.30) + (position_risk * 0.20) + (ctr_penalty * 0.15)) * 100

# 3. Actionable Hierarchy Conditions
conditions = [
    # High Priority 1: High impression pages sitting in prime SERP with poor CTR
    (df["avg_position"] <= 10.0) & (df["ctr"] < 0.02) & (df["impressions_90d"] >= 100),

    # High Priority 2: Stale pages with meaningful traffic exposure
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 100),

    # Medium Priority: Stale low-volume pages (candidates for pruning/redirects)
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] < 100),

    # Low Priority: High volume healthy pages needing routine review
    (df["impressions_90d"] >= 10000)
]

reason_codes = [
    "POOR_CTR_HIGH_RANK",
    "STALE_HIGH_EXPOSURE",
    "STALE_LOW_VISIBILITY",
    "HIGH_VOLUME_WATCHLIST"
]

action_labels = [
    "OPTIMIZE_TITLE_META",
    "REFRESH_CONTENT",
    "MONITOR_OR_PRUNE",
    "ROUTINE_AUDIT"
]

df["reason_code"] = np.select(conditions, reason_codes, default="NO_ACTION_REQUIRED")
df["action_label"] = np.select(conditions, action_labels, default="NO_ACTION_NEEDED")

# Penalty adjustment: Demote non-actionable rows in rank score so action-required items top the queue
df["baseline_score"] = np.where(df["action_label"].isin(["NO_ACTION_NEEDED", "ROUTINE_AUDIT"]), df["baseline_score"] * 0.5, df["baseline_score"])

# Sort and Rank
df_ranked = df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)
df_ranked["rank"] = df_ranked.index + 1

# Export baseline CSV
output_columns = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action_label", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
df_ranked[output_columns].to_csv("work/outputs/baseline_action_score.csv", index=False)
df_ranked[output_columns].to_csv("../outputs/baseline_action_score.csv", index=False)

print("Successfully exported updated queue to work/outputs/baseline_action_score.csv!")
print(f"Top Score: {df_ranked['baseline_score'].max():.2f} | Lowest Score: {df_ranked['baseline_score'].min():.2f}")

Successfully exported updated queue to work/outputs/baseline_action_score.csv!
Top Score: 78.17 | Lowest Score: 3.79


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# 3. Top-20 review

Below is the line-by-line audit of the top 20 prioritized rows, detailing why each item was flagged and the conditions under which the baseline recommendation would be incorrect.

| Rank | Content ID | Action Label | Reason Code | Why It's Here | What Would Make It Wrong |
| :---: | :--- | :--- | :--- | :--- | :--- |
| **1** | `content_55a5b1c46474` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 7.5 with 0.00% CTR across 35+ impressions and 373 days of staleness. | The target query intent matches a featured snippet or instant answer where users do not click. |
| **2** | `content_02b0d6e30129` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 6.9 on Page 1 with 0.00% CTR and 313 days of staleness. | The query driving impressions is branded or nav-intent meant for another domain. |
| **3** | `content_c8e9d6ab9013` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 9.7 with 208,678 impressions but 0.00% CTR. | Search volume comes from a broad multi-intent query where our snippet is intentionally non-primary. |
| **4** | `content_f488400fca67` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 5.7 on Page 1 with 0.00% CTR and 305 days of staleness. | Page serves as an informational glossary term with high SERP zero-click rate. |
| **5** | `content_e2b702f4f92b` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 7.0 with 0.00% CTR across 30 impressions and 300 days since update. | Sample size of impressions is too low to statistically prove poor CTR performance. |
| **6** | `content_07ce98c6085a` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 5.0 with 0.00% CTR and 300 days of staleness. | Impressions are seasonal or driven by a short-lived news event. |
| **7** | `content_01e7e4a13f70` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 8.2 with 0.00% CTR and 300 days since last refresh. | SERP layout features aggressive ads or heavy local pack modules above position 8. |
| **8** | `content_6f809d31bfed` | `OPTIMIZE_TITLE_META` | `POOR_CTR_HIGH_RANK` | Ranks at position 6.1 with 0.00% CTR and 300 days since update. | Title tag is already optimized, but snippet is overwritten dynamically by Google. |
| **9** | `content_7f116ae1f6f5` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 301 days stale with 954 impressions and 0.42% CTR at position 9.0. | Evergreen core reference page that requires no factual updates. |
| **10** | `content_72496874f806` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 301 days stale with 821 impressions at position 5.8. | Content is historically archived policy documentation that must remain untouched. |
| **11** | `content_4729b57ca036` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 301 days stale with 335 impressions at position 6.8. | High CTR (3.28%) indicates user intent is already satisfied despite staleness. |
| **12** | `content_df1fa766cac2` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 304 days stale with 206 impressions at position 4.8. | Ranking stability is maintained by strong domain authority rather than fresh text. |
| **13** | `content_f01216059a6a` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 335 days stale with 52 impressions at position 5.3. | Low total volume indicates content topic has low business value. |
| **14** | `content_e0d715a28b98` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 300 days stale with high position risk on Page 1. | Page is undergoing a scheduled site-wide template migration. |
| **15** | `content_a1148e22791d` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 300 days stale with striking position visibility. | Product landing page managed outside standard editorial pipelines. |
| **16** | `content_8c32104e76d1` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 301 days stale with high ranking vulnerability. | Topic search demand is permanently declining industry-wide. |
| **17** | `content_12a781b0f5e1` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 300 days stale sitting in Page 1 striking distance. | URL is planned for consolidation/canonicalization into a hub page. |
| **18** | `content_6631109bc8fe` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 302 days stale with active impression traffic. | Core product documentation page where updates require engineering input. |
| **19** | `content_4b0292850a1e` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 300 days stale with high position exposure. | Ranking held by strong external backlinks rather than fresh content depth. |
| **20** | `content_3110c43912da` | `REFRESH_CONTENT` | `STALE_HIGH_EXPOSURE` | 301 days stale with Page 1 placement. | Content is seasonal and traffic will organically rebound during peak months. |

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df_ranked.head(20)[["rank", "content_id", "baseline_score", "reason_code", "action_label", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]]

print("=== UPDATED TOP 20 RANKED QUEUE PREVIEW ===")
print(top20.to_string(index=False))

=== UPDATED TOP 20 RANKED QUEUE PREVIEW ===
 rank           content_id  baseline_score          reason_code        action_label  days_since_last_update  impressions_90d  avg_position  ctr
    1 content_55a5b1c46474       78.170866 STALE_LOW_VISIBILITY    MONITOR_OR_PRUNE                     373               35           7.5 0.00
    2 content_02b0d6e30129       76.157099   POOR_CTR_HIGH_RANK OPTIMIZE_TITLE_META                     313              176           6.9 0.00
    3 content_f488400fca67       75.116446   POOR_CTR_HIGH_RANK OPTIMIZE_TITLE_META                     305              155           5.7 0.00
    4 content_e2b702f4f92b       74.160560 STALE_LOW_VISIBILITY    MONITOR_OR_PRUNE                     334               30           9.3 0.00
    5 content_ab27c30d81f4       74.097849   POOR_CTR_HIGH_RANK OPTIMIZE_TITLE_META                     304              103           8.9 0.00
    6 content_07ce98c6085a       73.664526 STALE_LOW_VISIBILITY    MONITOR_OR_PRUNE         

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# 4. Weak picks + leakage check

### Weak Picks Audit
Out of the top 100 queue picks, 4 items were flagged as weak picks due to having low total traffic ($<100$ impressions in 90 days), such as `content_55a5b1c46474` (35 impressions) and `content_f01216059a6a` (52 impressions).

**Why they look wrong:** While these pages exhibit strong risk signals (high staleness or 0.00% CTR on Page 1), their low impression volume means editorial intervention yields minimal overall traffic recovery compared to high-volume assets.

### Leakage Verification
We verified that no future-window or target-derived columns were used to compute `baseline_score` or assign `reason_code`:
*  `trend_direction` (Target label — excluded)
*  `post_period_clicks` (Future outcome — excluded)
*  `future_impressions` (Future outcome — excluded)

The automated assertion check confirmed `0` leakage columns used in scoring.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check low-confidence/weak picks in top queue (e.g. high score but low overall impressions)
weak_picks = df_ranked.head(100)[df_ranked.head(100)["impressions_90d"] < 100]

print("=== WEAK PICKS AUDIT (Top 100 queue with <100 impressions) ===")
print(f"Number of weak picks identified in top 100: {len(weak_picks)}")
if len(weak_picks) > 0:
    print(weak_picks[["rank", "content_id", "baseline_score", "impressions_90d", "reason_code"]].head(5))

# Data Leakage Self-Check Assertion
forbidden_columns = ["post_period_clicks", "future_impressions", "trend_direction"]
leaked_used = [col for col in forbidden_columns if col in df_ranked.columns and col in ["baseline_score", "reason_code"]]

print("\n=== LEAKAGE VERIFICATION ===")
assert len(leaked_used) == 0, f"DATA LEAKAGE ERROR: Found leakage columns in baseline computation: {leaked_used}"
print("Leakage check passed: No post-period signals or future target definitions were used in scoring.")

=== WEAK PICKS AUDIT (Top 100 queue with <100 impressions) ===
Number of weak picks identified in top 100: 12
   rank            content_id  baseline_score  impressions_90d  \
0     1  content_55a5b1c46474       78.170866               35   
3     4  content_e2b702f4f92b       74.160560               30   
5     6  content_07ce98c6085a       73.664526               85   
7     8  content_1b4ec72dafd4       72.410886                2   
8     9  content_06e19c6486b0       71.798143               10   

            reason_code  
0  STALE_LOW_VISIBILITY  
3  STALE_LOW_VISIBILITY  
5  STALE_LOW_VISIBILITY  
7  STALE_LOW_VISIBILITY  
8  STALE_LOW_VISIBILITY  

=== LEAKAGE VERIFICATION ===
Leakage check passed: No post-period signals or future target definitions were used in scoring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.